In [ ]:
import subprocess
import os
import time
import re
import threading
import datetime
import sys
import glob

# --- KONFIGURASI USER & PASSWORD ---
USERNAME = "zavin"
PASSWORD = "Zavin123"
SSH_PORT = 2222
AFISLA_LOG = "afisla.log"

current_relay_port = None
sshd_process = None
afisla_process = None

def cleanup_old_sessions():
    """Membersihkan sesi dan file log lama."""
    print("\n🧹 === Memulai Pembersihan Total Sesi Lama ===")
    services = ["cloudflared", "afisla", "lt", "serveo", "sshd"]
    for s in services:
        subprocess.run(["sudo", "pkill", "-f", s], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    logs = ["cf.log", "afisla.log", "sshd_config_custom"]
    for l in logs:
        if os.path.exists(l):
            try:
                os.remove(l)
            except Exception:
                pass
    time.sleep(3)
    print("✅ Sesi lama bersih.")

def check_and_install_dependencies():
    """Memeriksa dan menginstal tools/dependensi yang dibutuhkan."""
    print("[+] Memeriksa dependensi sistem...")

    packages = ["openssh-server", "netcat-openbsd", "curl", "sudo"]
    needed_pkgs = []

    for pkg in packages:
        res = subprocess.run(["dpkg", "-s", pkg], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if res.returncode != 0:
            needed_pkgs.append(pkg)

    if needed_pkgs:
        print(f"[+] Menginstall paket yang hilang: {', '.join(needed_pkgs)}")
        subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
        subprocess.run(["sudo", "apt-get", "install", "-y", "-qq"] + needed_pkgs, check=True)

    if subprocess.run(["which", "afisla"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
        print("[+] Menginstall Afisla Tunnel Client...")
        os.system("curl -fsSL https://afisla.web.id/install.sh | bash")

    print("[✔] Semua dependensi siap.")

def setup_user():
    """Membuat user non-root dan mengatur password."""
    print(f"[+] Menyiapkan user '{USERNAME}'...")

    res = subprocess.run(["id", USERNAME], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if res.returncode != 0:
        subprocess.run(["sudo", "useradd", "-m", "-s", "/bin/bash", USERNAME], check=True)
        subprocess.run(["sudo", "usermod", "-aG", "sudo", USERNAME], check=True)
        print(f"[✔] User '{USERNAME}' berhasil dibuat.")

    # Set password user
    p1 = subprocess.Popen(["echo", f"{USERNAME}:{PASSWORD}"], stdout=subprocess.PIPE)
    subprocess.run(["sudo", "chpasswd"], stdin=p1.stdout, check=True)
    p1.stdout.close()

    # Set password root
    p2 = subprocess.Popen(["echo", f"root:{PASSWORD}"], stdout=subprocess.PIPE)
    subprocess.run(["sudo", "chpasswd"], stdin=p2.stdout, check=True)
    p2.stdout.close()

    print(f"[✔] Password untuk '{USERNAME}' & 'root' diset ke '{PASSWORD}'.")

def setup_sshd():
    """Membuat konfig sshd_custom dan menjalankan daemon sshd."""
    global sshd_process
    print("[+] Mengonfigurasi SSH Server...")

    ssh_config_content = f"""Port {SSH_PORT}
PermitRootLogin yes
PasswordAuthentication yes
PubkeyAuthentication no
ChallengeResponseAuthentication no
UsePAM yes
LogLevel VERBOSE
ClientAliveInterval 30
ClientAliveCountMax 3
TCPKeepAlive yes
Subsystem sftp /usr/lib/openssh/sftp-server
"""
    config_file = os.path.abspath("sshd_config_custom")
    with open(config_file, "w") as f:
        f.write(ssh_config_content)

    os.makedirs("/var/run/sshd", exist_ok=True)
    os.makedirs("/run/sshd", exist_ok=True)
    subprocess.run(["sudo", "chmod", "0755", "/var/run/sshd"], check=False)

    # Hapus host key lama jika ada
    for key_file in glob.glob("/etc/ssh/ssh_host_*"):
        try:
            os.remove(key_file)
        except Exception:
            pass

    subprocess.run(["sudo", "ssh-keygen", "-A"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)

    # Jalankan daemon SSHD
    sshd_process = subprocess.Popen(['sudo', '/usr/sbin/sshd', '-D', '-f', config_file],
                                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    print(f"✅ SSH Server berhasil berjalan di port {SSH_PORT}.")

def start_afisla():
    """Jalankan Afisla Tunnel Client."""
    global afisla_process
    print("[+] Menjalankan Afisla Tunnel Client...")
    afisla_log_file = open(AFISLA_LOG, "w")
    afisla_process = subprocess.Popen(['afisla', 'client', '--port-local', str(SSH_PORT)],
                                      stdout=afisla_log_file, stderr=subprocess.STDOUT)

def monitor_services():
    """Thread monitoring untuk auto-restart dan deteksi port relay."""
    global sshd_process, afisla_process, current_relay_port
    config_file = os.path.abspath("sshd_config_custom")

    while True:
        time.sleep(10)

        # Anti-Disconnect untuk terminal Colab
        sys.stdout.write('\x00')
        sys.stdout.flush()

        # Monitor & Restart SSHD jika mati
        if sshd_process and sshd_process.poll() is not None:
            sshd_process = subprocess.Popen(['sudo', '/usr/sbin/sshd', '-D', '-f', config_file],
                                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # Monitor Afisla Log & Extract Relay Port
        if os.path.exists(AFISLA_LOG):
            with open(AFISLA_LOG, "r") as f:
                log = f.read()
                match = re.search(r"(?:port|relay)[:\s]+(\d{4,5})", log, re.IGNORECASE) or re.search(r"(\d{5})", log)
                if match:
                    new_port = match.group(1)
                    if new_port != current_relay_port:
                        current_relay_port = new_port
                        print("\n" + "="*70)
                        print(f"[STABLE AFISLA TUNNEL READY]")
                        print(f"User     : {USERNAME}")
                        print(f"Password : {PASSWORD}")
                        print(f"Port Relay: {current_relay_port}")
                        print("\nJalankan perintah berikut di Terminal Lokal Anda:")
                        print(f"ssh -o ServerAliveInterval=30 -o PreferredAuthentications=password -o ProxyCommand='nc relay.afisla.web.id {current_relay_port}' {USERNAME}@127.0.0.1 -p {SSH_PORT}")
                        print("="*70 + "\n")

if __name__ == "__main__":
    cleanup_old_sessions()
    check_and_install_dependencies()
    setup_user()
    setup_sshd()
    start_afisla()

    # Jalankan monitoring di background thread
    threading.Thread(target=monitor_services, daemon=True).start()

    print("[+] Services started with Verbose Logging & Afisla TCP Relay.")

    # Main loop
    while True:
        time.sleep(120)
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Heartbeat: Services Running...")



🧹 === Memulai Pembersihan Total Sesi Lama ===
✅ Sesi lama bersih.
[+] Memeriksa dependensi sistem...
[+] Menginstall paket yang hilang: netcat-openbsd
[+] Menginstall Afisla Tunnel Client...
[✔] Semua dependensi siap.
[+] Menyiapkan user 'zavin'...
[✔] User 'zavin' berhasil dibuat.
[✔] Password untuk 'zavin' & 'root' diset ke 'Zavin123'.
[+] Mengonfigurasi SSH Server...
✅ SSH Server berhasil berjalan di port 2222.
[+] Menjalankan Afisla Tunnel Client...
[+] Services started with Verbose Logging & Afisla TCP Relay.
 
[STABLE AFISLA TUNNEL READY]
User     : zavin
Password : Zavin123
Port Relay: 30023

Jalankan perintah berikut di Terminal Lokal Anda:
ssh -o ServerAliveInterval=30 -o PreferredAuthentications=password -o ProxyCommand='nc relay.afisla.web.id 30023' zavin@127.0.0.1 -p 2222

          [05:17:31] Heartbeat: Services Running...
      

KeyboardInterrupt: 